# 🏗️ InfraPlan — Fine-tune Phi-3 Mini on Infrastructure Data

**Runtime: GPU (T4) — Runtime → Change runtime type → T4 GPU**

This notebook fine-tunes Microsoft Phi-3 Mini 3.8B on your scraped infrastructure project data.

## Steps:
1. Install dependencies
2. Upload `training_data.jsonl` from the scraper
3. Fine-tune with LoRA (fits in free T4 GPU)
4. Push to HuggingFace Hub
5. Copy the model URL to your backend `.env`

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q unsloth
!pip install -q transformers datasets peft trl accelerate bitsandbytes huggingface_hub
print('✅ Dependencies installed')

In [ ]:
# ── Step 2: Upload your training data ─────────────────────────────────────────
from google.colab import files
print('Upload training_data.jsonl from the scraper folder...')
uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# ── Step 3: Load and inspect data ─────────────────────────────────────────────
import json

data = []
with open('training_data.jsonl') as f:
    for line in f:
        line = line.strip()
        if line:
            data.append(json.loads(line))

print(f'📊 Total training examples: {len(data)}')
print(f'\nSample:')
print(json.dumps(data[0], indent=2)[:500])

In [ ]:
# ── Step 4: Format for Phi-3 ──────────────────────────────────────────────────
def format_prompt(example):
    return {
        'text': f"""<|system|>
You are an expert Indian infrastructure project planner. Decompose projects into detailed JSON execution plans.<|end|>
<|user|>
{example['instruction']}
{example['input']}<|end|>
<|assistant|>
{example['output']}<|end|>"""
    }

from datasets import Dataset
dataset = Dataset.from_list([format_prompt(d) for d in data])

# Split train/val
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print(f'Sample text:\n{train_dataset[0]["text"][:400]}')

In [ ]:
# ── Step 5: Load Phi-3 Mini with Unsloth (2x faster, fits T4) ─────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Phi-3-mini-4k-instruct',
    max_seq_length=2048,
    dtype=None,           # Auto-detect
    load_in_4bit=True,    # 4-bit quantization to fit T4 GPU
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print('✅ Model loaded with LoRA adapters')
model.print_trainable_parameters()

In [ ]:
# ── Step 6: Fine-tune ─────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='epoch',
        output_dir='./infraplan-phi3-checkpoints',
        report_to='none',
        seed=42,
    ),
)

print('🚀 Starting fine-tuning... (~15-25 min on T4)')
trainer_stats = trainer.train()
print(f'✅ Training complete! Loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# ── Step 7: Test the model ────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_prompt = """<|system|>
You are an expert Indian infrastructure project planner.<|end|>
<|user|>
Decompose this infrastructure project into a structured execution plan.
Project: Mumbai Coastal Road Extension Phase 2
Location: Mumbai, Maharashtra
Sector: Transportation
Budget: ₹4200 Cr
Duration: 36 months<|end|>
<|assistant|>
"""

inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=500, temperature=0.3, do_sample=True)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the response
response = result.split('<|assistant|>')[-1].strip()
print('Model output:')
print(response[:800])

In [ ]:
# ── Step 8: Push to HuggingFace Hub ──────────────────────────────────────────
from huggingface_hub import login

# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = input('Paste your HuggingFace token (from huggingface.co/settings/tokens): ').strip()
HF_USERNAME = input('Your HuggingFace username: ').strip()

login(token=HF_TOKEN)

MODEL_NAME = f'{HF_USERNAME}/infraplan-phi3'

print(f'Pushing to {MODEL_NAME}...')
model.push_to_hub(MODEL_NAME, token=HF_TOKEN)
tokenizer.push_to_hub(MODEL_NAME, token=HF_TOKEN)

print(f'''
✅ Model pushed!

Now add this to your backend/.env:

HF_MODEL_URL=https://api-inference.huggingface.co/models/{MODEL_NAME}
HF_TOKEN={HF_TOKEN}

Then restart the backend server.
''')

In [ ]:
# ── Optional: Save locally to download ───────────────────────────────────────
model.save_pretrained('./infraplan-phi3-local')
tokenizer.save_pretrained('./infraplan-phi3-local')

import shutil
shutil.make_archive('./infraplan-phi3-local', 'zip', './infraplan-phi3-local')

from google.colab import files
files.download('./infraplan-phi3-local.zip')
print('Model downloaded!')